# Spark 4.1 + Kafka connectivity check

This notebook assumes you started services on the same docker compose network:

```bash
docker compose -f docker-compose-spark41.yml -f docker-compose-kafka.yml -f docker-compose-notebooks.yml up -d --build
```

- Spark master: `spark://spark-master-41:7078`
- Kafka: `kafka:29092`


In [1]:
import os
from pyspark.sql import SparkSession

spark_master = os.environ.get("SPARK_MASTER", "spark://spark-master-41:7078")
kafka_bootstrap = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "kafka:29092")
topic = os.environ.get("KAFKA_TOPIC", "events")

spark = (
    SparkSession.builder
    .appName("spark41-kafka-connectivity-1")
    .master(spark_master)
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark master:", spark_master)
print("Kafka bootstrap:", kafka_bootstrap)
print("Topic:", topic)


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7265b93d-cc9b-4d22-b6ee-703fe411b231;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.0 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0

Spark version: 4.1.0
Spark master: spark://spark-master-41:7078
Kafka bootstrap: kafka:29092
Topic: events


In [2]:
# Batch read check (fails fast if topic/broker is wrong)
df = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap)
    .option("subscribe", topic)
    .option("startingOffsets", "earliest")
    .load()
)

df.selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value").show(10, truncate=False)


[Stage 0:>                                                          (0 + 1) / 1]

+----+-----+
|key |value|
+----+-----+
|NULL|hello|
+----+-----+



In [4]:
spark

In [3]:
spark.sparkContext.master


'spark://spark-master-41:7078'

In [4]:
df.selectExpr("CAST(value AS STRING)").count()

1

In [4]:
# spark.stop()

In [5]:
conf = dict(spark.sparkContext.getConf().getAll())
for k in [
  "spark.eventLog.enabled",
  "spark.eventLog.dir",
  "spark.history.fs.logDirectory",
  "spark.app.id",
]:
    print(k, conf.get(k))


spark.eventLog.enabled true
spark.eventLog.dir /opt/spark-data/logs
spark.history.fs.logDirectory /opt/spark-data/logs
spark.app.id app-20260428183901-0002


In [6]:
import os, glob
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("spark-defaults.conf exists:", os.path.exists("/opt/spark/conf/spark-defaults.conf"))
print("eventlog dirs:", glob.glob("/opt/spark-data/logs/eventlog_v2_*")[-5:])


SPARK_HOME: /opt/spark
spark-defaults.conf exists: True
eventlog dirs: ['/opt/spark-data/logs/eventlog_v2_app-20260427070801-0000', '/opt/spark-data/logs/eventlog_v2_local-1777281459070', '/opt/spark-data/logs/eventlog_v2_local-1777282092307', '/opt/spark-data/logs/eventlog_v2_local-1777273496437', '/opt/spark-data/logs/eventlog_v2_local-1777374657396']


In [7]:
conf = dict(spark.sparkContext.getConf().getAll())
for k in ["spark.eventLog.enabled","spark.eventLog.dir","spark.history.fs.logDirectory","spark.app.id"]:
    print(k, conf.get(k))


spark.eventLog.enabled true
spark.eventLog.dir /opt/spark-data/logs
spark.history.fs.logDirectory /opt/spark-data/logs
spark.app.id app-20260428183901-0002
